# الدرس 02 - استكشاف إطار عمل وكيل مايكروسوفت

يُعد **إطار عمل وكيل مايكروسوفت (MAF)** إطارًا موحدًا لبناء وكلاء الذكاء الاصطناعي. يوفر بنية نظيفة وقابلة للتكوين تتكون من أربعة مكونات أساسية:

- **العميل** – يتصل بنقطة نهاية نموذج الذكاء الاصطناعي ويتولى التواصل
- **الوكيل** – يغلف العميل بالتعليمات وتعريفات الأدوات
- **الأدوات** – توسع قدرات الوكيل بوظائف مخصصة يمكن للنموذج استدعاؤها
- **الجلسة** – تحافظ على سجل المحادثة للتفاعلات متعددة الأدوار

في هذا الدرس، سنبني **وكيل حجز سفر** يتحقق من توفر الوجهة باستخدام هذه المفاهيم.


## الإعداد


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## فهم بنية إطار الوكيل

يتبع إطار عمل وكيل Microsoft بنية متعددة الطبقات:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **العميل** – يتصل `AzureAIProjectAgentProvider` بنشر Azure OpenAI. يتولى المصادقة، تنسيق الطلب، وتحليل الاستجابة.
2. **الوكيل** – يتم إنشاؤه من العميل عبر `provider.create_agent()`، يدمج الوكيل الوصول إلى النموذج مع التعليمات (موجه النظام) والأدوات.
3. **الأدوات** – دوال Python مزينة بـ `@tool` يمكن للوكيل استدعاؤها لأداء إجراءات أو استرجاع بيانات.
4. **الجلسة** – كائن `AgentSession` (يتم إنشاؤه عبر `agent.create_session()`) يخزن سجل المحادثة، مما يتيح الحوار متعدد الأدوار حيث يتذكر الوكيل السياق السابق.

دعونا نبني كل طبقة خطوة بخطوة.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## إضافة أدوات باستخدام المزخرف @tool

تتيح الأدوات للوكلاء اتخاذ إجراءات تتجاوز فقط توليد النص. يقوم المزخرف `@tool` بتحويل دالة بايثون عادية إلى شيء يمكن للوكيل استدعاؤه.

النقاط الرئيسية:
- استخدم `Annotated[type, "description"]` حتى يفهم النموذج كل معلمة.
- تصبح سلسلة التوثيق هي وصف الأداة التي يراها النموذج.
- تعني `approval_mode="never_require"` أن الأداة تعمل تلقائيًا دون تأكيد من المستخدم.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## إنشاء وكيل مع أدوات

الآن نقوم بدمج العميل، التعليمات، والأدوات في وكيل. تعمل `instructions` كإشارة نظام — فهي تحدد شخصية الوكيل وسلوكه.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## المحادثات متعددة الأدوار مع الجلسات

تحتفظ `AgentSession` (التي يتم إنشاؤها عبر `agent.create_session()`) بسجل لجميع الرسائل في المحادثة. من خلال تمرير نفس الجلسة إلى كل استدعاء `agent.run()`، يمكن للوكيل الوصول إلى تاريخ المحادثة الكامل والرجوع إلى الرسائل السابقة.

نمرر `tools=[check_destination_availability]` حتى يتمكن الوكيل من استدعاء مدقق التوفر الخاص بنا خلال كل دور.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## ملخص

في هذا الدرس، استكشفت الأعمدة الأربعة لإطار عمل وكيل مايكروسوفت:

| المفهوم | ما تعلمته |
|---------|------------------|
| **العميل** | `AzureAIProjectAgentProvider` يتصل بـ Azure OpenAI باستخدام مصادقة تعتمد على الاعتماديات |
| **الوكيل** | `provider.create_agent()` يجمع بين اتصال النموذج مع التعليمات واسم |
| **الأدوات** | مزخرف `@tool` يعرض دوال بايثون ليتم استدعاؤها من قبل الوكيل |
| **الجلسة** | `agent.create_session()` يحافظ على سجل المحادثة عبر عدة دورات |

تتحد هذه المكونات الأساسية معًا لإنشاء وكلاء يمكنهم إجراء محادثات طبيعية، واستدعاء الدوال الخارجية، والحفاظ على السياق — وهو الأساس لأنماط الوكلاء المتقدمة في الدروس القادمة.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**إخلاء المسؤولية**:  
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بالرغم من سعينا للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاعتماد على الترجمة البشرية المهنية. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
